# 🔎 Build a Document Retriever Search Engine

Build a search engine over two kinds of documents - a Wikipedia sample (JSON lines) and a research paper (PDF) - and
compare retrieval strategies on it. There is no answer generation yet: every query returns ranked documents.

**Reference:** [RAG Systems Essentials (Analytics Vidhya)](https://courses.analyticsvidhya.com/courses/take/rag-systems-essentials/lessons/60148017-hands-on-deep-dive-into-rag-evaluation-metrics-generator-metrics-i)

## Learning Objectives
In this notebook, you will learn:
1. **Load JSON documents** - turn a Wikipedia JSONL dump into LangChain `Document`s with metadata
2. **Contextual chunking** - prepend an LLM-written context to every PDF chunk (Anthropic's contextual retrieval idea)
3. **Vector database** - embed documents into Chroma, persist it to disk and load it back
4. **Similarity and multi-query retrieval** - rank by cosine similarity, then let an LLM rewrite the query several ways
5. **Contextual compression** - extract or filter retrieved documents with an LLM
6. **Chained retrieval pipeline** - similarity search, then an LLM filter, then a cross-encoder reranker

## Prerequisites
- Embeddings, vector stores and retrievers (repository 05, retrieval and RAG) - background reading
- Dependencies installed with `uv sync` in this project's folder
- `OPENAI_API_KEY` in this project's `.env` (OpenAI embeddings and `gpt-4o-mini`)
- Part 6 downloads the open-source `BAAI/bge-reranker-large` cross-encoder (about 1.3 GB) from Hugging Face on its first run

---
## 📦 Step 1: Environment Setup

Load `OPENAI_API_KEY` from this project's `.env`, then create the chat model used for contextual chunking, multi-query rewriting and compression.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import warnings

from dotenv import find_dotenv, load_dotenv

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY from this project's .env
load_dotenv(find_dotenv(usecwd=True))

print("✅ Environment ready")

In [ ]:
# ============================================================================
# LLM INITIALIZATION: gpt-4o-mini for chunk contexts, query rewriting and compression
# ============================================================================
from aied_helpers import get_openai_llm

# temperature=0 is the factory default, as in the original lesson
llm = get_openai_llm(model="gpt-4o-mini")

print(f"✅ LLM initialized: {llm.model_name}")

---
## 📄 Part 1: Load JSON Documents from a Wikipedia Dump

`data/wikidata_rag_demo.jsonl` holds one Wikipedia article per line. `JSONLoader` reads each line as raw JSON text
(`text_content=False`); we then parse it and build one `Document` per article, with its paragraphs joined into the
content and the title, id and source kept as metadata.

In [ ]:
# ============================================================================
# LOAD JSONL: One raw Document per line of the Wikipedia sample
# ============================================================================
from langchain_community.document_loaders import JSONLoader

loader = JSONLoader(file_path='../data/wikidata_rag_demo.jsonl',
                    jq_schema='.',
                    text_content=False,
                    json_lines=True)
wiki_docs = loader.load()

In [ ]:
len(wiki_docs)

In [ ]:
print(wiki_docs[0].page_content)

In [ ]:
# ============================================================================
# PROCESS: Parse each line and build a Document with metadata
# ============================================================================
import json
from langchain_core.documents import Document

# Initialize an empty list to store processed Document objects
wiki_docs_processed = []

# Iterate through each document loaded from the JSONLoader
for doc in wiki_docs:
    # Parse the JSON content from the page_content attribute
    doc = json.loads(doc.page_content)
    
    # Extract relevant metadata fields for each document
    metadata = {
        "title": doc['title'],      # The title of the Wikipedia article
        "id": doc['id'],            # The unique identifier for the article
        "source": "Wikipedia"       # The source of the document
    }
    
    # Concatenate all paragraphs into a single string for the document content
    data = ' '.join(doc['paragraphs'])
    
    # Create a LangChain Document object with the combined content and metadata, and add it to the list
    wiki_docs_processed.append(Document(page_content=data, metadata=metadata))

In [ ]:
# the first three processed documents (the full list has one entry per article)
wiki_docs_processed[:3]

In [ ]:
print(wiki_docs_processed[1500])

---
## 🧩 Part 2: Contextual Chunks for PDF Documents

Here we borrow inspiration from Anthropic's [contextual retrieval](https://www.anthropic.com/news/contextual-retrieval) strategy which involves create a contextual summary for each chunk and adding it to the chunk before storing in the vector database.

![](https://i.imgur.com/cjnB831.png)

#### Create Chunk Contexts for Contextual Retrieval

![](https://i.imgur.com/LRhKHzk.png)

In [ ]:
# ============================================================================
# LOAD + SPLIT: The Attention paper, in 3,500-character chunks
# ============================================================================
# load PDF files with langchain
from langchain_community.document_loaders import PyMuPDFLoader

loader = PyMuPDFLoader("../data/1706.03762v7.pdf")
doc_pages = loader.load()

from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=3500,
                                          chunk_overlap=0)
doc_chunks = splitter.split_documents(doc_pages)

In [ ]:
len(doc_chunks)

In [ ]:
# the actual research paper
big_doc = '\n'.join([doc.page_content for doc in doc_chunks])

In [ ]:
len(big_doc.split(' '))

In [ ]:
# ============================================================================
# CHUNK CONTEXT: An LLM chain that situates one chunk inside the whole paper
# ============================================================================
# create a chat prompt
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


def generate_chunk_context(document, chunk):

    chunk_process_prompt = """You are an AI assistant specializing in research paper analysis.
                            Your task is to provide brief, relevant context for a chunk of text
                            based on the following research paper.

                            Here is the research paper:
                            <paper>
                            {paper}
                            </paper>

                            Here is the chunk we want to situate within the whole document:
                            <chunk>
                            {chunk}
                            </chunk>

                            Provide a concise context (3-4 sentences max) for this chunk,
                            considering the following guidelines:

                            - Give a short succinct context to situate this chunk within the overall document
                            for the purposes of improving search retrieval of the chunk.
                            - Answer only with the succinct context and nothing else.
                            - Context should be mentioned like 'Focuses on ....'
                            do not mention 'this chunk or section focuses on...'

                            Context:
                        """

    prompt_template = ChatPromptTemplate.from_template(chunk_process_prompt)

    agentic_chunk_chain = (prompt_template
                                |
                            llm
                                |
                            StrOutputParser())

    context = agentic_chunk_chain.invoke({'paper': document, 'chunk': chunk})

    return context

In [ ]:
print(doc_chunks[5].page_content)

In [ ]:
generate_chunk_context(big_doc, doc_chunks[5].page_content)

### Load and Process PDF Documents

`create_contextual_chunks` applies the same idea to every chunk of a file and prepends each generated context to its
chunk. Every call sends the whole paper to the LLM, so the cost grows with paper length times chunk count; that is why
only the *Attention Is All You Need* paper (`1706.03762v7.pdf`) is processed here.

In [ ]:
# ============================================================================
# CONTEXTUAL CHUNKS: Load, split and contextualise one PDF file
# ============================================================================
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

def create_contextual_chunks(file_path):

    print('Loading pages:', file_path)
    loader = PyMuPDFLoader(file_path)
    doc_pages = loader.load()

    print('Chunking pages:', file_path)
    splitter = RecursiveCharacterTextSplitter(chunk_size=3500,
                                              chunk_overlap=0)
    doc_chunks = splitter.split_documents(doc_pages)

    print('Generating contextual chunks:', file_path)
    original_doc = '\n'.join([doc.page_content for doc in doc_chunks])
    contextual_chunks = []
    for chunk in doc_chunks:
        context = generate_chunk_context(original_doc, chunk.page_content)
        contextual_chunks.append(Document(page_content=context+'\n'+chunk.page_content,
                                          metadata=chunk.metadata))
    print('Finished processing:', file_path)
    print()
    return contextual_chunks

In [ ]:
from glob import glob

pdf_files = glob('../data/*.pdf')
pdf_files

In [ ]:
paper_docs = []
for fp in pdf_files:
    # only the Attention paper: every chunk's prompt carries the whole paper, so contextualising
    # the 75-page GPT-3 paper (2005.14165v4.pdf) as well would cost many times more
    if '1706.03762' in fp:
        paper_docs.extend(create_contextual_chunks(fp))

In [ ]:
len(paper_docs)

In [ ]:
paper_docs[0]

In [ ]:
len(wiki_docs_processed)

In [ ]:
total_docs = wiki_docs_processed + paper_docs
len(total_docs)

---
## 🗄️ Part 3: Store the Documents in a Chroma Vector DB

One of the most common ways to store and search over unstructured data is to embed it and store the resulting embedding vectors, and then at query time to embed the unstructured query and retrieve the embedding vectors that are 'most similar' to the embedded query. A vector database takes care of storing embedded data and performing vector search for you.

### Chroma Vector DB

[Chroma](https://docs.trychroma.com/getting-started) is a AI-native open-source vector database focused on developer productivity and happiness. Chroma is licensed under Apache 2.0.

### Open AI Embedding Models

LangChain enables us to access Open AI embedding models which include the newest models: a smaller and highly efficient `text-embedding-3-small` model, and a larger and more powerful `text-embedding-3-large` model.

In [ ]:
# ============================================================================
# EMBEDDINGS: OpenAI text-embedding-3-small
# ============================================================================
from langchain_openai import OpenAIEmbeddings

# details here: https://openai.com/blog/new-embedding-models-and-api-updates
openai_embed_model = OpenAIEmbeddings(model='text-embedding-3-small')

### Create a Vector DB and persist on disk

Here we initialize a connection to a Chroma vector DB client, and also we want to save to disk, so we simply initialize the Chroma client and pass the directory where we want the data to be saved to.

In [ ]:
# ============================================================================
# VECTOR DB: Embed all documents into a persistent Chroma collection
# ============================================================================
from pathlib import Path

from langchain_chroma import Chroma

# create vector DB of docs and embeddings - takes < 30 s
# from_documents on an existing folder would store every document a second time,
# so the vector DB is built only once - delete ./my_db to rebuild it
if Path("./my_db").exists():
    print("✅ ./my_db already exists - the next cell loads it (delete the folder to rebuild)")
else:
    chroma_db = Chroma.from_documents(documents=total_docs,
                                      collection_name='my_db',
                                      embedding=openai_embed_model,
                                      # need to set the distance function to cosine else it uses euclidean by default
                                      # check https://docs.trychroma.com/guides#changing-the-distance-function
                                      collection_metadata={"hnsw:space": "cosine"},
                                      persist_directory="./my_db")
    print(f"✅ Indexed {len(total_docs)} documents into ./my_db")

### Load Vector DB from disk

This is just to show once you have a vector database on disk you can just load and create a connection to it anytime

In [ ]:
# load from disk
chroma_db = Chroma(persist_directory="./my_db",
                   collection_name='my_db',
                   embedding_function=openai_embed_model)

In [ ]:
chroma_db

---
## 🎯 Part 4: Similarity and Multi-Query Retrieval

Here we will explore the following retrieval strategies on our Vector Database:

- Similarity or Ranking based Retrieval
- Multi Query Retrieval
- Contextual Compression Retrieval
- Chained Retrieval Pipeline

Parts 5 and 6 cover the last two strategies.

### Similarity or Ranking based Retrieval

We use cosine similarity here and retrieve the top 5 similar documents based on the user input query

In [ ]:
similarity_retriever = chroma_db.as_retriever(search_type="similarity",
                                              search_kwargs={"k": 5})

In [ ]:
# ============================================================================
# DISPLAY HELPER: Print metadata and the first 1,000 characters of each document
# ============================================================================
from IPython.display import display, Markdown

def display_docs(docs):
    for doc in docs:
        print('Metadata:', doc.metadata)
        print('Content Brief:')
        display(Markdown(doc.page_content[:1000]))
        print()

In [ ]:
query = "what is machine learning?"
top_docs = similarity_retriever.invoke(query)
display_docs(top_docs)

In [ ]:
query = "what is ML?"
top_docs = similarity_retriever.invoke(query)
display_docs(top_docs)

In [ ]:
query = "what is the difference between transformers and vision transformers?"
top_docs = similarity_retriever.invoke(query)
display_docs(top_docs)

In [ ]:
query = "what is a cnn?"
top_docs = similarity_retriever.invoke(query)
display_docs(top_docs)

In [ ]:
query = "what is deep learning?"
top_docs = similarity_retriever.invoke(query)
display_docs(top_docs)

In [ ]:
query = "what is nlp?"
top_docs = similarity_retriever.invoke(query)
display_docs(top_docs)

### Multi Query Retrieval

Retrieval may produce different results with subtle changes in query wording, or if the embeddings do not capture the semantics of the data well. Prompt engineering / tuning is sometimes done to manually address these problems, but can be tedious.

The [`MultiQueryRetriever`](https://api.python.langchain.com/en/latest/retrievers/langchain.retrievers.multi_query.MultiQueryRetriever.html) automates the process of prompt tuning by using an LLM to generate multiple queries from different perspectives for a given user input query. For each query, it retrieves a set of relevant documents and takes the unique union across all queries to get a larger set of potentially relevant documents.

In [ ]:
# ============================================================================
# MULTI-QUERY RETRIEVER: The LLM writes several versions of each query
# ============================================================================
from langchain_classic.retrievers.multi_query import MultiQueryRetriever
# Set logging for the queries
import logging

similarity_retriever = chroma_db.as_retriever(search_type="similarity",
                                              search_kwargs={"k": 5})

mq_retriever = MultiQueryRetriever.from_llm(
    retriever=similarity_retriever, llm=llm
)

logging.basicConfig()
# so we can see what queries are generated by the LLM
logging.getLogger("langchain_classic.retrievers.multi_query").setLevel(logging.INFO)

The `INFO` log lines under each query show the rewritten queries the LLM generated.

In [ ]:
query = "what is a cnn?"
top_docs = mq_retriever.invoke(query)
display_docs(top_docs)

In [ ]:
query = "what is nlp?"
top_docs = mq_retriever.invoke(query)
display_docs(top_docs)

In [ ]:
query = "what is ML?"
top_docs = mq_retriever.invoke(query)
display_docs(top_docs)

---
## 🗜️ Part 5: Contextual Compression Retrieval

The information most relevant to a query may be buried in a document with a lot of irrelevant text. Passing that full document through your application can lead to more expensive LLM calls and poorer responses.

Contextual compression is meant to fix this. The idea is simple: instead of immediately returning retrieved documents as-is, you can compress them using the context of the given query, so that only the relevant information is returned.

This compression can happen in the form of:

- Remove parts of the content of retrieved documents which are not relevant to the query. This is done by extracting only relevant parts of the document to the given query

- Filter out documents which are not relevant to the given query but do not remove content from the document

Here we wrap our multi-query retriever with a `ContextualCompressionRetriever`. Then we'll add an `LLMChainExtractor`, which will iterate over the initially returned documents and extract from each only the content that is relevant to the query.

In [ ]:
# ============================================================================
# COMPRESSION: LLMChainExtractor keeps only the query-relevant text
# ============================================================================
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor


# extracts from each document only the content that is relevant to the query
compressor = LLMChainExtractor.from_llm(llm=llm)

# retrieves the documents similar to query and then applies the compressor
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=mq_retriever
)

In [ ]:
query = "what is ML?"
top_docs = compression_retriever.invoke(query)
display_docs(top_docs)

In [ ]:
query = "what is a cnn?"
top_docs = compression_retriever.invoke(query)
display_docs(top_docs)

In [ ]:
query = "what is nlp?"
top_docs = compression_retriever.invoke(query)
display_docs(top_docs)

In [ ]:
query = "what is clustering?"
top_docs = compression_retriever.invoke(query)
display_docs(top_docs)

In [ ]:
query = "what is a neural network?"
top_docs = compression_retriever.invoke(query)
display_docs(top_docs)

The `LLMChainFilter` is slightly simpler but more robust compressor that uses an LLM chain to decide which of the initially retrieved documents to filter out and which ones to return, without manipulating the document contents.

In [ ]:
# ============================================================================
# FILTER: LLMChainFilter keeps or drops whole documents
# ============================================================================
from langchain_classic.retrievers.document_compressors import LLMChainFilter

#  decides which of the initially retrieved documents to filter out and which ones to return
_filter = LLMChainFilter.from_llm(llm=llm)

# retrieves the documents similar to query and then applies the filter
compression_retriever = ContextualCompressionRetriever(
    base_compressor=_filter, base_retriever=mq_retriever
)

In [ ]:
query = "what is ML?"
top_docs = compression_retriever.invoke(query)
display_docs(top_docs)

In [ ]:
query = "what is NLP?"
top_docs = compression_retriever.invoke(query)
display_docs(top_docs)

In [ ]:
query = "what is a neural network?"
top_docs = compression_retriever.invoke(query)
display_docs(top_docs)

In [ ]:
query = "what is a cnn?"
top_docs = compression_retriever.invoke(query)
display_docs(top_docs)

---
## ⛓️ Part 6: Chained Retrieval Pipeline

This strategy uses a chain of multiple retrievers sequentially to get to the most relevant documents. The following is the flow

Similarity Retrieval → Compression Filter → Reranker Model Retrieval

![](http://i.imgur.com/77pXxLu.gif)

The reranker is the open-source `BAAI/bge-reranker-large` cross-encoder. The first run downloads it (about 1.3 GB) from Hugging Face; later runs use the local cache.

In [ ]:
# ============================================================================
# CHAINED RETRIEVER: similarity -> LLM filter -> cross-encoder reranker (top 3)
# ============================================================================
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_classic.retrievers.document_compressors import LLMChainFilter
from langchain_classic.retrievers import ContextualCompressionRetriever

# Retriever 1 - simple cosine distance based retriever
similarity_retriever = chroma_db.as_retriever(search_type="similarity",
                                              search_kwargs={"k": 5})

#  decides which of the initially retrieved documents to filter out and which ones to return
_filter = LLMChainFilter.from_llm(llm=llm)
# Retriever 2 - retrieves the documents similar to query and then applies the filter
compressor_retriever = ContextualCompressionRetriever(
    base_compressor=_filter, base_retriever=similarity_retriever
)

# download an open-source reranker model - BAAI/bge-reranker-large (~1.3 GB, cached after the first run)
reranker = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-large")
reranker_compressor = CrossEncoderReranker(model=reranker, top_n=3)
# Retriever 3 - Uses a Reranker model to rerank retrieval results from the previous retriever
final_retriever = ContextualCompressionRetriever(
    base_compressor=reranker_compressor, base_retriever=compressor_retriever
)

In [ ]:
query = "what is ML?"
top_docs = final_retriever.invoke(query)
display_docs(top_docs)

In [ ]:
query = "what is a neural network?"
top_docs = final_retriever.invoke(query)
display_docs(top_docs)

In [ ]:
query = "what is a transformer model?"
top_docs = final_retriever.invoke(query)
display_docs(top_docs)

In [ ]:
query = "what is nlp?"
top_docs = final_retriever.invoke(query)
display_docs(top_docs)

In [ ]:
query = "what is clustering?"
top_docs = final_retriever.invoke(query)
display_docs(top_docs)

In [ ]:
query = "what is a vision transformer"
top_docs = final_retriever.invoke(query)
display_docs(top_docs)

In [ ]:
query = "what is statistics"
top_docs = final_retriever.invoke(query)
display_docs(top_docs)

---
## 📝 Summary

In this notebook, we learned:

### 1. Load JSON Documents
- A JSONL dump becomes LangChain `Document`s once each line is parsed and its fields are mapped to content and metadata.

### 2. Contextual Chunking
- Prepending a short, LLM-written context to every chunk tells the retriever where the chunk sits in its document.
- Each context call sends the whole document, so this is worth it for a few key documents, not a whole corpus.

### 3. Vector Database
- `Chroma.from_documents` embeds and persists in one step; `Chroma(persist_directory=...)` reconnects later.
- Build once: re-running `from_documents` on the same folder duplicates every document.

### 4. Similarity and Multi-Query Retrieval
- Similarity search ranks by cosine similarity; `MultiQueryRetriever` unions the results of several LLM-rewritten queries.

### 5. Contextual Compression
- `LLMChainExtractor` trims documents to the relevant text; `LLMChainFilter` keeps or drops whole documents.

### 6. Chained Retrieval Pipeline
- Retrievers compose: a cheap similarity search, an LLM filter and a cross-encoder reranker each narrow the results.

### Next Steps
- **`02_rag_system_for_question_answering.ipynb`**: adds the generation step - retrieved chunks go into a prompt and the LLM answers the question.